In [ ]:
# ============================================================
# DEPENDENCY INSTALLATION
# ============================================================
# Run this cell once before the rest of the notebook.
!pip install tslearn ta-lib kneed

In [ ]:
# ============================================================
# PHASE 1: DATA INGESTION & PRE-PROCESSING
# ============================================================
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

# --- 1. Ingestion ---
df = pd.read_csv('top_companies_20y_daily_combined.csv')
print(f"Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

# --- 2. Date Parsing & Chronological Sort ---
# Critical: sort by Ticker first, then Date to prevent timeline scrambling.
df['Date'] = pd.to_datetime(df['Date'])
df['Ticker'] = df['Ticker'].str.strip()   # kill hidden whitespace from CSV
df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# --- 3. Duplicate Removal ---
initial_rows = df.shape[0]
df = df.drop_duplicates()
print(f"Duplicate rows removed: {initial_rows - df.shape[0]}")

# --- 4. Missing Value Imputation (Forward then Backward Fill) ---
# FIX: Use groupby().transform() instead of groupby().apply() to avoid
# the pandas DeprecationWarning about include_groups.
# Grouping by Ticker is mandatory to prevent data leakage across companies.
price_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close']
for col in price_cols:
    df[col] = df.groupby('Ticker')[col].transform(lambda x: x.ffill().bfill())

# --- 5. Anomaly Detection (Zero or Negative Prices) ---
# A stock price cannot be zero or negative. Convert to NaN and re-bridge.
# FIX: Use .map() instead of deprecated .applymap().
df[price_cols] = df[price_cols].map(lambda x: np.nan if x <= 0 else x)
for col in price_cols:
    df[col] = df.groupby('Ticker')[col].transform(lambda x: x.ffill().bfill())

# --- 6. Final Integrity Check ---
print(f"Final missing values remaining: {df.isnull().sum().sum()}")
print(f"Tickers available: {df['Ticker'].nunique()}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

In [ ]:
# ============================================================
# PHASE 2: FEATURE ENGINEERING & TARGET DEFINITION (with train/val split)
# ============================================================

# ==================== CONTROL PANEL ====================
# OPTION A: Manually select tickers.
SELECTED_TICKERS = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'TSLA']

# OPTION B: Random subset. Set to an integer to activate; None uses OPTION A.
NUM_RANDOM_TICKERS = None
# =======================================================

all_tickers = df['Ticker'].unique()
if NUM_RANDOM_TICKERS is not None:
    SELECTED_TICKERS = random.sample(list(all_tickers), NUM_RANDOM_TICKERS)
    print(f"Randomly selected {NUM_RANDOM_TICKERS} tickers: {SELECTED_TICKERS}")
else:
    print(f"Using manually selected tickers: {SELECTED_TICKERS}")

# FIX: Filter FIRST, then engineer features on the subset.
df_filtered = df[df['Ticker'].isin(SELECTED_TICKERS)].copy()
print(f"Rows for selected tickers: {len(df_filtered)}")

# --- BASELINE: SMA Spread (equal temporal weight) ---
df_filtered['SMA_10'] = df_filtered.groupby('Ticker')['Close'].transform(
    lambda x: x.rolling(window=10).mean())
df_filtered['SMA_50'] = df_filtered.groupby('Ticker')['Close'].transform(
    lambda x: x.rolling(window=50).mean())
df_filtered['Trend_Signal_SMA'] = df_filtered['SMA_10'] - df_filtered['SMA_50']

# --- FRESHNESS VARIANT: EMA Spread (recent action weighted higher) ---
df_filtered['EMA_10'] = df_filtered.groupby('Ticker')['Close'].transform(
    lambda x: x.ewm(span=10, adjust=False).mean())
df_filtered['EMA_50'] = df_filtered.groupby('Ticker')['Close'].transform(
    lambda x: x.ewm(span=50, adjust=False).mean())
df_filtered['Trend_Signal_EMA'] = df_filtered['EMA_10'] - df_filtered['EMA_50']

# --- TARGET: Binary label (1=Up, 0=Down) 3 days post-window ---
df_filtered['Future_Close_3D'] = df_filtered.groupby('Ticker')['Close'].transform(
    lambda x: x.shift(-3))

# Drop NaNs generated by rolling windows and the 3-day shift
df_final = df_filtered.dropna().reset_index(drop=True)

# ─── SPLIT into TRAIN (2006-2019) and VALIDATION (2020-2026) ───────────
df_train = df_final[df_final['Date'].dt.year <= 2019]
df_val   = df_final[df_final['Date'].dt.year >= 2020]

print(f"\nClean rows total: {len(df_final)}")
print(f"Training rows   : {len(df_train)}  (2006–2019)")
print(f"Validation rows : {len(df_val)}    (2020–2026)")
print(f"NaN check: {df_final.isnull().sum().sum()}")

In [ ]:
# ============================================================
# PHASE 3: SEQUENTIAL WINDOWING (train & val separately)
# Slices the 2D time series into 3D tensors.
# Both SMA and EMA tensors are built simultaneously for fair comparison.
# ============================================================
WINDOW_SIZE = 10

def build_windows(data_df):
    X_sma      = []   # SMA momentum wave
    X_ema      = []   # EMA momentum wave
    X_raw_ohlc = []   # Raw OHLC candles (for Phase 7 TA-Lib)
    y_targets  = []   # Binary future outcome (1=Up, 0=Down)
    dates_info = []   # (Ticker, end_date) metadata for Phase 6 year breakdown

    for ticker, group in data_df.groupby('Ticker'):
        sma_sig  = group['Trend_Signal_SMA'].values
        ema_sig  = group['Trend_Signal_EMA'].values
        opens    = group['Open'].values
        highs    = group['High'].values
        lows     = group['Low'].values
        closes   = group['Close'].values
        future   = group['Future_Close_3D'].values
        dates    = group['Date'].dt.strftime('%Y-%m-%d').values

        if len(sma_sig) < WINDOW_SIZE:
            continue

        for i in range(len(sma_sig) - WINDOW_SIZE):
            X_sma.append(sma_sig[i : i + WINDOW_SIZE])
            X_ema.append(ema_sig[i : i + WINDOW_SIZE])
            X_raw_ohlc.append(np.column_stack((
                opens[i:i+WINDOW_SIZE],
                highs[i:i+WINDOW_SIZE],
                lows[i:i+WINDOW_SIZE],
                closes[i:i+WINDOW_SIZE]
            )))
            current_close = closes[i + WINDOW_SIZE - 1]
            future_val    = future[i + WINDOW_SIZE - 1]
            y_targets.append(1 if future_val > current_close else 0)
            dates_info.append((ticker, dates[i + WINDOW_SIZE - 1]))

    # Convert to final 3D tensors
    return (np.array(X_sma).reshape(-1, WINDOW_SIZE, 1),
            np.array(X_ema).reshape(-1, WINDOW_SIZE, 1),
            np.array(X_raw_ohlc),
            np.array(y_targets),
            dates_info)

# Build training and validation tensors separately
X_sma_train, X_ema_train, X_ohlc_train, y_train, dates_train = build_windows(df_train)
X_sma_val,   X_ema_val,   X_ohlc_val,   y_val,   dates_val   = build_windows(df_val)

print(f"Train SMA tensor : {X_sma_train.shape}   Up ratio: {y_train.mean():.2%}")
print(f"Train EMA tensor : {X_ema_train.shape}")
print(f"Train OHLC tensor: {X_ohlc_train.shape}")
print(f"\nVal SMA tensor   : {X_sma_val.shape}     Up ratio: {y_val.mean():.2%}")
print(f"Val EMA tensor   : {X_ema_val.shape}")
print(f"Val OHLC tensor  : {X_ohlc_val.shape}")

In [ ]:
# ============================================================
# PHASE 3b: PCA DIMENSIONALITY REDUCTION (fitted on train)
# Reduces each 10-dimensional window to fewer principal components.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from sklearn.decomposition import PCA

# --- Step 1: Normalise tensors (fit on train, transform train & val) ---
# Two separate scaler instances: re-fitting one scaler on EMA after SMA would
# overwrite the SMA statistics, causing transform(X_sma_val) to apply wrong
# mean/variance — a subtle but real data leakage bug.
sma_scaler = TimeSeriesScalerMeanVariance()
ema_scaler = TimeSeriesScalerMeanVariance()
X_sma_norm_train = sma_scaler.fit_transform(X_sma_train)
X_ema_norm_train = ema_scaler.fit_transform(X_ema_train)
X_sma_norm_val   = sma_scaler.transform(X_sma_val)
X_ema_norm_val   = ema_scaler.transform(X_ema_val)
print(f"Normalised - SMA train: {X_sma_norm_train.shape}  SMA val: {X_sma_norm_val.shape}")

# --- Step 2: Flatten 3D → 2D for PCA input ---
X_sma_flat_train = X_sma_norm_train.reshape(X_sma_norm_train.shape[0], -1)
X_sma_flat_val   = X_sma_norm_val.reshape(X_sma_norm_val.shape[0], -1)
X_ema_flat_train = X_ema_norm_train.reshape(X_ema_norm_train.shape[0], -1)
X_ema_flat_val   = X_ema_norm_val.reshape(X_ema_norm_val.shape[0], -1)
print(f"Flattened train SMA: {X_sma_flat_train.shape}")

# --- Step 3: Fit PCA retaining 95% of variance on train, transform val ---
pca_sma = PCA(n_components=0.95, random_state=42)
X_sma_pca_train = pca_sma.fit_transform(X_sma_flat_train)
X_sma_pca_val   = pca_sma.transform(X_sma_flat_val)

pca_ema = PCA(n_components=0.95, random_state=42)
X_ema_pca_train = pca_ema.fit_transform(X_ema_flat_train)
X_ema_pca_val   = pca_ema.transform(X_ema_flat_val)

n_comp_sma = pca_sma.n_components_
n_comp_ema = pca_ema.n_components_
var_sma = pca_sma.explained_variance_ratio_.sum() * 100
var_ema = pca_ema.explained_variance_ratio_.sum() * 100
print(f"\nSMA: {WINDOW_SIZE} dims → {n_comp_sma} PCs  ({var_sma:.1f}% variance retained)")
print(f"EMA: {WINDOW_SIZE} dims → {n_comp_ema} PCs  ({var_ema:.1f}% variance retained)")

# --- Step 4: Separate 2-component PCA strictly for visualisation (fit on train) ---
pca_2d = PCA(n_components=2, random_state=42)
X_sma_2d_train = pca_2d.fit_transform(X_sma_flat_train)
X_sma_2d_val   = pca_2d.transform(X_sma_flat_val)
X_ema_2d_train = pca_2d.transform(X_ema_flat_train)  # same plane as SMA (useful for comparison)
X_ema_2d_val   = pca_2d.transform(X_ema_flat_val)
vis_var = pca_2d.explained_variance_ratio_ * 100
print(f"2D vis: PC1={vis_var[0]:.1f}%  PC2={vis_var[1]:.1f}%  total={vis_var.sum():.1f}%")

# --- Step 5: Scree plot + cumulative variance ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — individual explained variance per PC (SMA)
individual = pca_sma.explained_variance_ratio_ * 100
axes[0].bar(range(1, n_comp_sma + 1), individual,
            color='steelblue', edgecolor='white', linewidth=0.7)
axes[0].axhline(0, color='grey', linewidth=0.5)
for idx, val in enumerate(individual):
    axes[0].text(idx + 1, val + 0.3, f"{val:.1f}%",
                 ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0].set_xlabel("Principal Component", fontsize=11)
axes[0].set_ylabel("Variance Explained (%)", fontsize=11)
axes[0].set_title("Scree Plot — SMA Windows\n"
                  f"({n_comp_sma} PCs retain {var_sma:.1f}% variance)",
                  fontweight='bold', fontsize=12)
axes[0].set_xticks(range(1, n_comp_sma + 1))
axes[0].grid(True, linestyle='--', alpha=0.4, axis='y')

# Right — cumulative explained variance (SMA)
cumulative = np.cumsum(pca_sma.explained_variance_ratio_) * 100
axes[1].plot(range(1, n_comp_sma + 1), cumulative,
             'bo-', linewidth=2.2, markersize=8, label='Cumulative')
axes[1].axhline(95, color='tomato', linestyle='--', linewidth=1.5,
                label='95% threshold')
axes[1].axhline(99, color='orange', linestyle=':', linewidth=1.2,
                label='99% threshold')
axes[1].fill_between(range(1, n_comp_sma + 1), cumulative,
                     alpha=0.12, color='steelblue')
for idx, val in enumerate(cumulative):
    axes[1].text(idx + 1, val - 2.5, f"{val:.1f}%",
                 ha='center', va='top', fontsize=8, color='navy')
axes[1].set_xlabel("Number of Principal Components", fontsize=11)
axes[1].set_ylabel("Cumulative Variance Explained (%)", fontsize=11)
axes[1].set_title("Cumulative Variance — SMA Windows\n"
                  f"(95% threshold reached at PC {n_comp_sma})",
                  fontweight='bold', fontsize=12)
axes[1].set_xticks(range(1, n_comp_sma + 1))
axes[1].set_ylim(0, 105)
axes[1].legend(fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("pca_scree.png", dpi=200)
plt.show()
print("\nPCA complete. X_sma_pca_train used in Phase 4 for silhouette scoring.")
print("X_sma_2d_train/val passed to Phase 5 for cluster scatter plots.")

In [ ]:
# ============================================================
# PHASE 4: HYPERPARAMETER TUNING — ELBOW + SILHOUETTE DUAL SEARCH (on train)
# Auto-selects OPTIMAL_K without human bias.
# ============================================================
from tslearn.clustering import TimeSeriesKMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator

K_RANGE     = range(2, 16)
inertias    = []
silhouettes = []

print("\nRunning dual hyperparameter search (K=2..15, Euclidean) on training data...")
print(f"{'K':>4}  {'Inertia':>12}  {'Silhouette (PCA)':>18}")
print("-" * 38)

for k in K_RANGE:
    km = TimeSeriesKMeans(
        n_clusters=k, metric="euclidean",
        n_init=1, max_iter=30, random_state=42, verbose=False
    )
    # FIT ON TRAINING DATA ONLY
    labels = km.fit_predict(X_sma_norm_train)
    inertias.append(km.inertia_)
    # Silhouette computed in PCA space (noise removed) on TRAIN
    sil = silhouette_score(
        X_sma_pca_train, labels,
        sample_size=min(5000, len(X_sma_pca_train)),
        random_state=42
    )
    silhouettes.append(sil)
    print(f"{k:>4}  {km.inertia_:>12.2f}  {sil:>18.4f}")

# --- Elbow K via Kneedle ---
kl = KneeLocator(list(K_RANGE), inertias, curve="convex", direction="decreasing")
elbow_k = kl.knee if kl.knee else list(K_RANGE)[int(np.argmax(np.diff(np.diff(inertias)))) + 1]

# --- Silhouette K ---
sil_best_k = list(K_RANGE)[int(np.argmax(silhouettes))]

# --- Agreement logic ---
if elbow_k == sil_best_k:
    OPTIMAL_K  = elbow_k
    verdict    = f"Both methods agree → K = {OPTIMAL_K}"
    agree_flag = True
else:
    OPTIMAL_K  = round((elbow_k + sil_best_k) / 2)
    verdict    = (f"Elbow→K={elbow_k}, Silhouette→K={sil_best_k}"
                  f" → averaged to K={OPTIMAL_K}")
    agree_flag = False

print(f"\n{'='*50}")
print(f"  Elbow K        : {elbow_k}")
print(f"  Silhouette K   : {sil_best_k}")
print(f"  OPTIMAL K      : {OPTIMAL_K}  ({'agreement' if agree_flag else 'averaged'})")
print(f"  Verdict        : {verdict}")
print(f"{'='*50}")

# --- Dual-axis plot ---
fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()

ax1.plot(list(K_RANGE), inertias,    "bo-", linewidth=2, markersize=7, label="Inertia (↓)")
ax2.plot(list(K_RANGE), silhouettes, "rs--", linewidth=2, markersize=7, label="Silhouette (↑)")
ax1.axvline(elbow_k,    color="blue",  linestyle=":", alpha=0.7, label=f"Elbow K={elbow_k}")
ax2.axvline(sil_best_k, color="red",   linestyle=":", alpha=0.7, label=f"Silhouette K={sil_best_k}")
ax1.axvline(OPTIMAL_K,  color="green", linestyle="-", linewidth=2.5, label=f"Selected K={OPTIMAL_K}")

ax1.set_xlabel("Number of Clusters (K)", fontsize=12)
ax1.set_ylabel("Inertia", color="blue", fontsize=12)
ax2.set_ylabel("Silhouette Score", color="red", fontsize=12)
ax1.set_xticks(list(K_RANGE))
ax1.grid(True, linestyle="--", alpha=0.5)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)
plt.title(f"Phase 4: Dual Hyperparameter Tuning — Auto-selected K={OPTIMAL_K} (train)",
          fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("hyperparameter_tuning.png", dpi=200)
plt.show()
print(f"\nOptimal K = {OPTIMAL_K} — passed automatically into Phase 5.")

In [ ]:
# ============================================================
# PHASE 5: UNSUPERVISED PATTERN DISCOVERY — DTW K-MEANS
# Fits on training data, assigns validation data with .predict()
# OPTIMAL_K is inherited automatically from Phase 4.
# ============================================================

print(f"Using auto-selected K = {OPTIMAL_K}  (Elbow + Silhouette dual tuning)")

DTW_PARAMS = dict(
    n_clusters   = OPTIMAL_K,
    metric       = "dtw",
    init         = "k-means++",
    n_init       = 2,
    max_iter     = 10,
    verbose      = False,
    random_state = 42
)

# --- SMA clustering: fit on train, predict for val ---
print(f"Clustering SMA training data ({X_sma_norm_train.shape[0]} windows) with K={OPTIMAL_K}...")
dtw_km_sma = TimeSeriesKMeans(**DTW_PARAMS)
cluster_labels_sma_train = dtw_km_sma.fit_predict(X_sma_norm_train)
cluster_labels_sma_val   = dtw_km_sma.predict(X_sma_norm_val)
print(f"SMA train clusters: {np.unique(cluster_labels_sma_train)}")
print(f"SMA val clusters  : {np.unique(cluster_labels_sma_val)}")

# --- EMA clustering: fit on train, predict for val ---
print(f"\nClustering EMA training data ({X_ema_norm_train.shape[0]} windows) with K={OPTIMAL_K}...")
dtw_km_ema = TimeSeriesKMeans(**DTW_PARAMS)
cluster_labels_ema_train = dtw_km_ema.fit_predict(X_ema_norm_train)
cluster_labels_ema_val   = dtw_km_ema.predict(X_ema_norm_val)
print(f"EMA train clusters: {np.unique(cluster_labels_ema_train)}")
print(f"EMA val clusters  : {np.unique(cluster_labels_ema_val)}")

# --- Visualise SMA centroids (learned from training) ---
n_cols = 4
n_rows = int(np.ceil(OPTIMAL_K / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
fig.suptitle(
    f"Phase 5: {OPTIMAL_K}-Pattern Taxonomy — SMA Momentum Waves (DTW Centroids from Training)",
    fontsize=14, fontweight="bold"
)
axes = axes.flatten()
for i in range(OPTIMAL_K):
    centroid = dtw_km_sma.cluster_centers_[i].ravel()
    count    = np.sum(cluster_labels_sma_train == i)
    axes[i].plot(centroid, color="#1f77b4", linewidth=2.5)
    axes[i].set_title(f"Pattern {i}\n(n={count})", fontweight="bold", fontsize=10)
    axes[i].axhline(0, color="grey", linewidth=0.8, linestyle="--")
    axes[i].grid(True, linestyle="--", alpha=0.5)
    axes[i].set_xticks([]); axes[i].set_yticks([])
for j in range(OPTIMAL_K, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig("dtw_centroids_sma.png", dpi=200)
plt.show()
print("Centroid taxonomy saved.")

# --- PCA Cluster Scatter (Training & Validation) ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    f"Phase 5: PCA Cluster Scatter — DTW Assignments in 2D PCA Space\n"
    f"(PC1={vis_var[0]:.1f}%  PC2={vis_var[1]:.1f}% of SMA variance)",
    fontsize=14, fontweight='bold'
)

_CMAP = plt.get_cmap('tab10' if OPTIMAL_K <= 10 else 'tab20')

# Plot layout: [train_SMA, train_EMA; val_SMA, val_EMA]
for ax_row, (dataset_name, X_2d_sma, X_2d_ema, lbl_sma, lbl_ema) in enumerate([
    ('Training (2006-2019)', X_sma_2d_train, X_ema_2d_train, cluster_labels_sma_train, cluster_labels_ema_train),
    ('Validation (2020-2026)', X_sma_2d_val, X_ema_2d_val, cluster_labels_sma_val, cluster_labels_ema_val)
]):
    for ax_col, (variant_name, X_2d, labels) in enumerate([
        ('SMA Baseline', X_2d_sma, lbl_sma),
        ('EMA Freshness', X_2d_ema, lbl_ema)
    ]):
        ax = axes[ax_row, ax_col]
        N_PLOT = min(8000, len(X_2d))
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X_2d), N_PLOT, replace=False)
        xs, ys = X_2d[idx, 0], X_2d[idx, 1]
        cs = labels[idx]
        scatter = ax.scatter(xs, ys, c=cs, cmap=_CMAP,
                             vmin=0, vmax=OPTIMAL_K-1,
                             s=6, alpha=0.35, linewidths=0)
        # Plot centroids from training (projected into same PCA plane)
        km_model = dtw_km_sma if variant_name == 'SMA Baseline' else dtw_km_ema
        centroids_flat = km_model.cluster_centers_.reshape(OPTIMAL_K, -1)
        centroids_2d   = pca_2d.transform(centroids_flat)
        for c_id in range(OPTIMAL_K):
            ax.scatter(centroids_2d[c_id, 0], centroids_2d[c_id, 1],
                       c=[_CMAP(c_id / max(OPTIMAL_K-1, 1))],
                       s=180, marker='*', edgecolors='black', linewidths=0.8, zorder=5)
            ax.annotate(f"P{c_id}", (centroids_2d[c_id, 0], centroids_2d[c_id, 1]),
                        fontsize=8, fontweight='bold', ha='center', va='bottom',
                        xytext=(0, 8), textcoords='offset points')
        ax.set_xlabel(f"PC1 ({vis_var[0]:.1f}%)", fontsize=10)
        ax.set_ylabel(f"PC2 ({vis_var[1]:.1f}%)", fontsize=10)
        ax.set_title(f"{dataset_name} — {variant_name}  (n={N_PLOT:,} sampled)", fontweight='bold', fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.colorbar(scatter, ax=ax, ticks=range(OPTIMAL_K), label='Cluster ID').ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("pca_cluster_scatter.png", dpi=200)
plt.show()
print("PCA cluster scatter saved to 'pca_cluster_scatter.png'.")

In [ ]:
# ============================================================
# PHASE 6: EMPIRICAL ROBUSTNESS PROFILING (training set only)
# Win rates are grouped STRICTLY BY CALENDAR YEAR (2006-2019)
# to identify all-weather patterns vs. regime-dependent mirages.
# ============================================================

results_train = pd.DataFrame(dates_train, columns=['Ticker', 'Date'])
results_train['Date']        = pd.to_datetime(results_train['Date'])
results_train['Year']        = results_train['Date'].dt.year
results_train['Target']      = y_train
results_train['Cluster_SMA'] = cluster_labels_sma_train
results_train['Cluster_EMA'] = cluster_labels_ema_train

def yearly_win_rate_matrix(results_df, cluster_col, variant_label):
    """
    Build a Cluster x Year win-rate matrix.
    Isolates regime-dependent vs. robust patterns.
    """
    years    = sorted(results_df['Year'].unique())
    clusters = sorted(results_df[cluster_col].unique())

    rows = []
    for c in clusters:
        row = {'Cluster': c}
        c_mask = results_df[cluster_col] == c
        for y in years:
            subset = results_df[c_mask & (results_df['Year'] == y)]
            row[str(y)] = round(subset['Target'].mean() * 100, 1) if len(subset) > 0 else np.nan
        row['Overall_Win%'] = round(results_df[c_mask]['Target'].mean() * 100, 2)
        row['Size']         = int(c_mask.sum())
        rows.append(row)

    matrix = pd.DataFrame(rows).set_index('Cluster')
    matrix = matrix.sort_values('Overall_Win%', ascending=False)

    print(f"\n{'='*70}")
    print(f"  PHASE 6 — {variant_label} Robustness Matrix (Win Rate % by Year) — TRAINING SET")
    print(f"  Clusters above 55% across multiple years = regime-robust")
    print(f"{'='*70}")
    year_cols = [str(y) for y in years]
    print(matrix[year_cols + ['Overall_Win%', 'Size']].to_string(na_rep='—'))
    return matrix

matrix_sma = yearly_win_rate_matrix(results_train, 'Cluster_SMA', 'SMA Baseline')
matrix_ema = yearly_win_rate_matrix(results_train, 'Cluster_EMA', 'EMA Freshness Variant')
print("\nTraining robustness profiling complete.")

In [ ]:
# ============================================================
# PHASE 8: FINAL SYSTEM EVALUATION — SMA vs EMA (on validation set)
# Uses winning clusters identified from TRAINING.
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

# Build validation results dataframe
results_val = pd.DataFrame(dates_val, columns=['Ticker', 'Date'])
results_val['Date']        = pd.to_datetime(results_val['Date'])
results_val['Target']      = y_val
results_val['Cluster_SMA'] = cluster_labels_sma_val
results_val['Cluster_EMA'] = cluster_labels_ema_val

# NOTE: results_val_raw is also referenced in Phase 7 (Cell 8).
# If running cells in order: run this cell BEFORE Phase 7, or re-run Phase 7
# after this cell. Phase 7 uses results_val_raw only for ticker metadata.
results_val_raw = results_val.copy()

def evaluate_variant(results_df, cluster_col, matrix_train, label, threshold=55.0):
    """Evaluate on validation set using training‑discovered winning clusters."""
    winning_ids = list(matrix_train[matrix_train['Overall_Win%'] > threshold].index)
    y_pred = results_df[cluster_col].apply(lambda x: 1 if x in winning_ids else 0)
    y_true = results_df['Target']

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cov  = y_pred.mean()

    print(f"\n{'─'*52}")
    print(f"  {label} System  |  Threshold: >{threshold}%  |  K={OPTIMAL_K}")
    print(f"  Winning clusters (from train): {winning_ids}")
    print(f"  Signal coverage (val)       : {cov:.1%} ({y_pred.sum():,} of {len(y_pred):,} windows)")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Precision       : {prec:.4f}")
    print(f"  Recall          : {rec:.4f}")
    print(f"  F1 Score        : {f1:.4f}")
    print(f"{'─'*52}")
    return {'Variant': label, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1': f1, 'Coverage': cov}

sma_metrics = evaluate_variant(results_val, 'Cluster_SMA', matrix_sma, 'SMA Baseline')
ema_metrics = evaluate_variant(results_val, 'Cluster_EMA', matrix_ema, 'EMA Freshness')

# --- Side-by-side summary ---
comparison = pd.DataFrame([sma_metrics, ema_metrics]).set_index('Variant')
print("\n===== FINAL COMPARISON ON VALIDATION SET: SMA vs EMA =====")
print(comparison.to_string(float_format='{:.4f}'.format))

winner = comparison['F1'].idxmax()
print(f"\nConclusion: '{winner}' achieved superior F1 score on validation data.")

# ─── TRAIN SET EVALUATION (overfitting check) ────────────────────────────────
print("\n" + "="*52)
print("  TRAIN SET METRICS (sanity / overfitting check)")
print("="*52)
sma_train_metrics = evaluate_variant(results_train, 'Cluster_SMA', matrix_sma, 'SMA Baseline [TRAIN]')
ema_train_metrics = evaluate_variant(results_train, 'Cluster_EMA', matrix_ema, 'EMA Freshness [TRAIN]')

# ─── OVERFITTING SUMMARY ─────────────────────────────────────────────────────
print("\n" + "="*52)
print("  OVERFITTING ANALYSIS — Train F1 minus Validation F1")
print("  (close to 0 = good generalisation; large positive = overfit)")
print("="*52)
for variant, tr_m, val_m in [
    ('SMA Baseline',  sma_train_metrics, sma_metrics),
    ('EMA Freshness', ema_train_metrics, ema_metrics),
]:
    delta = tr_m['F1'] - val_m['F1']
    flag  = '✓ OK' if delta < 0.02 else ('⚠ Moderate' if delta < 0.06 else '✗ High overfit')
    print(f"  {variant:<18} Train F1={tr_m['F1']:.4f}  Val F1={val_m['F1']:.4f}  Δ={delta:+.4f}  {flag}")

print("\nPipeline complete. ✅")

In [1]:
# ============================================================
# PHASE 7: SUPERVISED CHARACTERISATION — TA-LIB VALIDATION
# Scans the raw OHLC windows of the VALIDATION set that
# belong to winning clusters (identified from training).
# ─────────────────────────────────────────────────────────
# NOTE: results_val_raw is created in Phase 8 (cell above).
# Cells are ordered so Phase 8 runs first — run top to bottom.
# ============================================================
import talib

WIN_RATE_THRESHOLD = 55.0   # clusters above this % are considered 'winning'

winning_sma_rows = matrix_sma[matrix_sma['Overall_Win%'] > WIN_RATE_THRESHOLD]
winning_sma_ids  = list(winning_sma_rows.index)
print(f"Winning SMA clusters (train) > {WIN_RATE_THRESHOLD}%: {winning_sma_ids}")

if not winning_sma_ids:
    print("No winning clusters found. Lower WIN_RATE_THRESHOLD or increase OPTIMAL_K.")
else:
    talib_patterns = {
        'Morning_Star':   talib.CDLMORNINGSTAR,
        'Engulfing':      talib.CDLENGULFING,
        'Hammer':         talib.CDLHAMMER,
        'Doji':           talib.CDLDOJI,
        'Shooting_Star':  talib.CDLSHOOTINGSTAR,
        'Harami':         talib.CDLHARAMI,
        'Three_White':    talib.CDL3WHITESOLDIERS,
    }

    pattern_matches = []
    # Iterate over validation SMA windows
    for idx, c_id in enumerate(cluster_labels_sma_val):
        if c_id not in winning_sma_ids:
            continue
        win = X_ohlc_val[idx]
        o = win[:, 0].astype(float)
        h = win[:, 1].astype(float)
        l = win[:, 2].astype(float)
        c = win[:, 3].astype(float)
        for name, func in talib_patterns.items():
            try:
                res = func(o, h, l, c)
                if res[-1] != 0:
                    pattern_matches.append({
                        'Cluster': c_id,
                        'Pattern': name,
                        'Signal':  int(res[-1]),   # +100=bullish, -100=bearish
                        'Ticker':  results_val_raw.iloc[idx]['Ticker']   # see note below
                    })
            except Exception:
                pass

    if pattern_matches:
        match_df = pd.DataFrame(pattern_matches)
        print("\n--- Phase 7: Pattern Count per Cluster (validation windows) ---")
        summary = match_df.groupby(['Cluster', 'Pattern']).size().unstack(fill_value=0)
        print(summary)

        print("\n--- Bullish (+100) vs Bearish (-100) signal breakdown ---")
        bull_bear = match_df.groupby(['Pattern', 'Signal']).size().unstack(fill_value=0)
        print(bull_bear)
    else:
        print("No textbook patterns detected in winning clusters of validation set.")
        print("This is itself an informative finding — DTW discovered non-textbook shapes.")

NameError: name 'matrix_sma' is not defined

In [ ]:
# ============================================================
# STREAMLIT DASHBOARD -- Launch Cell
# ============================================================
# Run this cell to launch the Streamlit UI.
# Requires: streamlit plotly tslearn scikit-learn kneed
#   pip install streamlit plotly tslearn scikit-learn kneed
# TA-Lib is optional; tick 'Skip TA-Lib scan' in the sidebar.
#
# Place pr_dashboard.py in the same folder as this notebook.
# Open  http://localhost:8501  in your browser after running.
# ============================================================

import os, subprocess, sys, pathlib

DASHBOARD_FILE = pathlib.Path('pr_dashboard finally.py')

if not DASHBOARD_FILE.exists():
    print('ERROR: pr_dashboard finally.py not found in:', os.getcwd())
    print('Copy pr_dashboard finally.py to the same folder as this notebook.')
else:
    print(f'Dashboard ready: {DASHBOARD_FILE.resolve()}')
    cmd = [
        sys.executable, '-m', 'streamlit', 'run', str(DASHBOARD_FILE),
        '--server.headless', 'true',
        '--browser.gatherUsageStats', 'false',
    ]
    print('\nLaunching... Open  http://localhost:8501  in your browser.')
    print('Press the STOP button in Jupyter to shut it down.\n')
    subprocess.run(cmd)

Dashboard ready: C:\Users\User\Desktop\pelajaran\pattern recog\DATASET\pr_dashboard finally.py

Launching... Open  http://localhost:8501  in your browser.
Press the STOP button in Jupyter to shut it down.



In [3]:
!streamlit run pr_dashboard_finaltrue.py 

^C
